# E8 - YOLO26s + SCConv at 768 px

This is a resolution-only follow-up to E5. The model starts from `yolo26s.pt`,
inserts the checked E5 `C3k2_SCConv` blocks at the runtime-selected P3/8 and
P4/16 neck nodes, and changes only `imgsz` from 640 to 768.

The dataset, class order, augmentation, optimizer, loss settings, batch size,
seed, validation split, and training protocol remain unchanged. This notebook
does not use the test split and keeps the standard Ultralytics training dataloader.

In [ ]:
%pip install -q ultralytics==8.4.155
%pip install -q gdown
!apt-get update -qq
!apt-get install -y unrar

In [ ]:
from google.colab import drive
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import ultralytics
import yaml
from PIL import Image
from ultralytics import YOLO

drive.mount('/content/drive', force_remount=True)
os.makedirs('/content/DetectedHelmets', exist_ok=True)
os.chdir('/content/DetectedHelmets')
data_root = Path('/content/DetectedHelmets/DataSet2')
drive_dataset_path = Path('/content/drive/MyDrive/DataSet2')
if drive_dataset_path.exists():
    if data_root.is_symlink():
        data_root.unlink()
    elif data_root.exists():
        print('Keeping existing local dataset directory; no files will be deleted.')
    else:
        data_root.symlink_to(drive_dataset_path, target_is_directory=True)
else:
    import gdown
    data_root.mkdir(parents=True, exist_ok=True)
    archive_path = Path('/content/DetectedHelmets/dataset.rar')
    file_id = '1LUUQLl5F3aczuny4As_PcTtqLIWTmZSJ'
    gdown.download(f'https://drive.google.com/uc?id={file_id}', str(archive_path), quiet=False, fuzzy=True)
    os.system(f'unrar x -o+ {archive_path} {data_root}/')

DATA_YAML_PATH = Path('/content/DetectedHelmets/DataSet2/DataSet2/data.yaml')
MODEL_NAME = 'yolo26s.pt'
ULTRALYTICS_VERSION = '8.4.155'
OUTPUT_DIR = Path('/content/drive/MyDrive/DetectedHelmets/runs/train')
EXPERIMENT_NAME = 'helmet_yolo26s_scconv_768_dataset2'
E5_EXPERIMENT_NAME = 'helmet_yolo26s_scconv_dataset2'
EXPERIMENT_DIR = OUTPUT_DIR / EXPERIMENT_NAME
BEST_MODEL_PATH = EXPERIMENT_DIR / 'weights' / 'best.pt'
E5_BEST_MODEL_PATH = OUTPUT_DIR / E5_EXPERIMENT_NAME / 'weights' / 'best.pt'
VALIDATION_SPLIT = 'val'

# The only experiment change relative to E5 is imgsz: 640 -> 768.
TRAIN_CONFIG = {
    'epochs': 50, 'imgsz': 768, 'batch': 16, 'device': 0, 'workers': 4,
    'optimizer': 'auto', 'seed': 0, 'deterministic': True, 'patience': 15,
    'pretrained': True, 'close_mosaic': 10, 'amp': True,
    'box': 7.5, 'cls': 0.5, 'dfl': 1.5,
    'weight_decay': 0.0005, 'momentum': 0.937,
    'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4,
    'degrees': 0.0, 'translate': 0.1, 'scale': 0.5,
    'shear': 0.0, 'perspective': 0.0,
    'flipud': 0.0, 'fliplr': 0.5,
    'mosaic': 1.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0,
}

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Ultralytics:', ultralytics.__version__)
assert torch.cuda.is_available(), 'E8 requires device=0 with a CUDA GPU.'
assert ultralytics.__version__ == ULTRALYTICS_VERSION
assert DATA_YAML_PATH.exists(), DATA_YAML_PATH
with DATA_YAML_PATH.open('r', encoding='utf-8') as file:
    data_config = yaml.safe_load(file)
raw_names = data_config.get('names', [])
if isinstance(raw_names, dict):
    raw_names = [raw_names[key] for key in sorted(raw_names, key=lambda item: int(item))]
CLASS_NAMES = list(raw_names)
EXPECTED_CLASS_NAMES = ['motorcycle', 'rider_with_helmet', 'rider_without_helmet']
assert data_config.get('nc') == 3
assert CLASS_NAMES == EXPECTED_CLASS_NAMES, CLASS_NAMES
assert TRAIN_CONFIG['imgsz'] == 768 and TRAIN_CONFIG['batch'] == 16
assert VALIDATION_SPLIT == 'val'
print('Dataset:', DATA_YAML_PATH)
print('Classes:', CLASS_NAMES)
print('E8 protocol:', TRAIN_CONFIG)
print('Output:', EXPERIMENT_DIR)

## 1. Runtime graph, exact E5 SCConv, and preflight checks

The target modules are selected from the live YOLO26s graph. The notebook
stops unless exactly one neck `C3k2` feeds the Detect head at each of P3/8
and P4/16. No other architecture module is inserted.

In [ ]:
base_model = YOLO(MODEL_NAME)
print('--- YOLO26s runtime architecture ---')
print(base_model.model)
base_model.info(verbose=True, imgsz=768)

def class_count(module, class_name):
    return sum(1 for item in module.modules() if item.__class__.__name__ == class_name)

def probe_c3k2(yolo_model, imgsz=768):
    records, handles = [], []
    def make_hook(index, module):
        def hook(_module, inputs, output):
            if not torch.is_tensor(output):
                return
            input_value = inputs[0] if inputs else None
            records.append({
                'index': index,
                'from': getattr(module, 'f', None),
                'input_shape': tuple(input_value.shape) if torch.is_tensor(input_value) else None,
                'output_shape': tuple(output.shape),
                'stride': imgsz // output.shape[-1],
            })
        return hook
    for index, module in enumerate(yolo_model.model.model):
        if module.__class__.__name__ == 'C3k2':
            handles.append(module.register_forward_hook(make_hook(index, module)))
    device = next(yolo_model.model.parameters()).device
    was_training = yolo_model.model.training
    yolo_model.model.eval()
    with torch.no_grad():
        yolo_model.model(torch.zeros(1, 3, imgsz, imgsz, device=device))
    if was_training:
        yolo_model.model.train()
    for handle in handles:
        handle.remove()
    return sorted(records, key=lambda item: item['index'])

c3k2_probe = probe_c3k2(base_model)
detect_entries = [
    (i, m) for i, m in enumerate(base_model.model.model)
    if m.__class__.__name__ in {'Detect', 'WorldDetect', 'YOLOEDetect'}
]
if not detect_entries:
    print(base_model.model)
    raise RuntimeError('Detect head not found.')
detect_index, detect_head = min(detect_entries, key=lambda item: item[0])
detect_from = getattr(detect_head, 'f', None)
if not isinstance(detect_from, (list, tuple)):
    print(base_model.model)
    raise RuntimeError('Detect input indices unavailable.')
detect_source_indices = {int(index) for index in detect_from}
upsample_indices = [
    i for i, m in enumerate(base_model.model.model)
    if m.__class__.__name__ == 'Upsample' and i < detect_index
]
if not upsample_indices:
    print(base_model.model)
    raise RuntimeError('Neck start could not be found.')
neck_start_index = min(upsample_indices)
neck_candidates = [
    record for record in c3k2_probe
    if neck_start_index < record['index'] < detect_index
    and record['index'] in detect_source_indices
    and record['stride'] in (8, 16)
]
selected_by_stride = {}
for stride in (8, 16):
    candidates = [record for record in neck_candidates if record['stride'] == stride]
    if len(candidates) != 1:
        print('Candidates:', candidates)
        print(base_model.model)
        raise RuntimeError(f'Expected exactly one neck C3k2 Detect input at stride {stride}.')
    selected_by_stride[stride] = candidates[0]
C3K2_SCCONV_EXPECTED_COUNT = 2
c3k2_specs = [
    {'level': f'P{stride // 8 + 2}/{stride}', **selected_by_stride[stride]}
    for stride in (8, 16)
]
print('Selected runtime C3k2 targets:', c3k2_specs)

def nearest_valid_group_num(c_num, requested):
    valid = [group for group in range(1, c_num + 1) if c_num % group == 0]
    return min(valid, key=lambda group: (abs(group - requested), -group))

class GroupBatchnorm2d(nn.Module):
    def __init__(self, c_num, group_num=4, eps=1e-5):
        super().__init__()
        self.group_num = nearest_valid_group_num(c_num, group_num)
        if self.group_num != group_num:
            print(f'GroupBatchnorm2d adjusted group_num {group_num} -> {self.group_num} for C={c_num}')
        self.weight = nn.Parameter(torch.ones(c_num, 1, 1))
        self.bias = nn.Parameter(torch.zeros(c_num, 1, 1))
        self.eps = eps

    def forward(self, x):
        batch, channels, height, width = x.shape
        grouped = x.reshape(batch, self.group_num, -1)
        mean = grouped.mean(2, keepdim=True)
        variance = grouped.var(2, keepdim=True, unbiased=False)
        normalized = ((grouped - mean) / torch.sqrt(variance + self.eps)).reshape(batch, channels, height, width)
        return normalized * self.weight + self.bias

class SRU(nn.Module):
    def __init__(self, oup_channels, group_num=4, gate_threshold=0.5):
        super().__init__()
        if oup_channels % 2:
            raise ValueError('SRU cross reconstruction requires even channels.')
        self.gn = GroupBatchnorm2d(oup_channels, group_num)
        self.gate_threshold = gate_threshold
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        gn_x = self.gn(x)
        gamma = self.gn.weight
        gamma_weight = (gamma / (gamma.sum() + torch.finfo(gamma.dtype).eps)).view(1, -1, 1, 1)
        weights = self.sigmoid(gn_x * gamma_weight)
        informative = torch.where(weights > self.gate_threshold, torch.ones_like(weights), weights)
        non_informative = torch.where(weights > self.gate_threshold, torch.zeros_like(weights), weights)
        return self.reconstruct(informative * x, non_informative * x)

    def reconstruct(self, x1, x2):
        half = x1.shape[1] // 2
        x11, x12 = torch.split(x1, [half, half], 1)
        x21, x22 = torch.split(x2, [half, half], 1)
        return torch.cat([x11 + x22, x12 + x21], 1)

class CRU(nn.Module):
    def __init__(self, op_channel, alpha=0.5, squeeze_ratio=2, group_size=2, group_kernel_size=3):
        super().__init__()
        self.up_channel = int(alpha * op_channel)
        self.low_channel = op_channel - self.up_channel
        self.up_squeeze = self.up_channel // squeeze_ratio
        self.low_squeeze = self.low_channel // squeeze_ratio
        if min(self.up_squeeze, self.low_squeeze) < 1 or self.up_squeeze % group_size or op_channel % group_size:
            raise ValueError('CRU channel configuration is incompatible.')
        self.squeeze1 = nn.Conv2d(self.up_channel, self.up_squeeze, 1, bias=False)
        self.squeeze2 = nn.Conv2d(self.low_channel, self.low_squeeze, 1, bias=False)
        self.GWC = nn.Conv2d(self.up_squeeze, op_channel, group_kernel_size, 1, group_kernel_size // 2, groups=group_size)
        self.PWC1 = nn.Conv2d(self.up_squeeze, op_channel, 1, bias=False)
        self.PWC2 = nn.Conv2d(self.low_squeeze, op_channel - self.low_squeeze, 1, bias=False)
        self.advavg = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        upper, lower = torch.split(x, [self.up_channel, self.low_channel], 1)
        upper, lower = self.squeeze1(upper), self.squeeze2(lower)
        upper_branch = self.GWC(upper) + self.PWC1(upper)
        lower_branch = torch.cat([self.PWC2(lower), lower], 1)
        fused = torch.cat([upper_branch, lower_branch], 1)
        fused = F.softmax(self.advavg(fused), dim=1) * fused
        first, second = torch.split(fused, fused.shape[1] // 2, 1)
        return first + second

SCONV_CONFIG = {
    'group_num': 4, 'gate_threshold': 0.5, 'alpha': 0.5,
    'squeeze_ratio': 2, 'group_size': 2, 'group_kernel_size': 3,
}

class ScConv(nn.Module):
    def __init__(self, op_channel, group_num=4, gate_threshold=0.5, alpha=0.5, squeeze_ratio=2, group_size=2, group_kernel_size=3):
        super().__init__()
        self.SRU = SRU(op_channel, group_num, gate_threshold)
        self.CRU = CRU(op_channel, alpha, squeeze_ratio, group_size, group_kernel_size)

    def forward(self, x):
        return self.CRU(self.SRU(x))

unit_input = torch.randn(2, 32, 16, 16, requires_grad=True)
unit_module = ScConv(32, **SCONV_CONFIG)
unit_output = unit_module(unit_input)
assert unit_output.shape == unit_input.shape
unit_output.mean().backward()
assert any(parameter.grad is not None for parameter in unit_module.parameters())
print('SCConv unit shape/gradient checks: OK')

## 2. C3k2_SCConv integration, 768 preflight, trainer, and checkpoint

The custom trainer overrides only model construction. It does not replace the
Ultralytics training dataloader. Batch 16 is mandatory; CUDA OOM is surfaced as
a stop condition instead of being converted into a different experiment.

In [ ]:
from ultralytics.nn.modules.block import Bottleneck

def wrapped_channels(wrapper):
    raw = getattr(wrapper, 'conv', wrapper)
    return int(raw.in_channels), int(raw.out_channels)

def wrapped_stride(wrapper):
    stride = getattr(getattr(wrapper, 'conv', wrapper), 'stride', 1)
    return (stride, stride) if isinstance(stride, int) else tuple(stride)

class Bottleneck_SCConv(nn.Module):
    def __init__(self, original):
        super().__init__()
        self.cv1 = original.cv1
        self.add = bool(getattr(original, 'add', False))
        input_channels, output_channels = wrapped_channels(original.cv2)
        self.scconv_applied = (
            wrapped_stride(original.cv2) == (1, 1)
            and input_channels == output_channels
            and input_channels % 2 == 0
        )
        self.cv2 = ScConv(input_channels, **SCONV_CONFIG) if self.scconv_applied else original.cv2

    def forward(self, x):
        output = self.cv2(self.cv1(x))
        return x + output if self.add else output

def replace_bottlenecks(module, stats):
    if module.__class__.__name__ == 'Bottleneck':
        replacement = Bottleneck_SCConv(module)
        stats['seen'] += 1
        stats['scconv'] += int(replacement.scconv_applied)
        stats['kept'] += int(not replacement.scconv_applied)
        return replacement
    for name, child in list(module.named_children()):
        replacement = replace_bottlenecks(child, stats)
        if replacement is not child:
            setattr(module, name, replacement)
    return module

class C3k2_SCConv(nn.Module):
    def __init__(self, original):
        super().__init__()
        self.c, self.cv1, self.cv2 = original.c, original.cv1, original.cv2
        self.stats = {'seen': 0, 'scconv': 0, 'kept': 0}
        self.m = replace_bottlenecks(original.m, self.stats)

    def forward(self, x):
        values = list(self.cv1(x).chunk(2, 1))
        values.extend(block(values[-1]) for block in self.m)
        return self.cv2(torch.cat(values, 1))

    def forward_split(self, x):
        values = list(self.cv1(x).split((self.c, self.c), 1))
        values.extend(block(values[-1]) for block in self.m)
        return self.cv2(torch.cat(values, 1))

def copy_graph_metadata(new, old, index, from_value):
    new.i = getattr(old, 'i', index)
    new.f = getattr(old, 'f', from_value)
    new.type = getattr(old, 'type', new.__class__.__name__)
    new.np = sum(parameter.numel() for parameter in new.parameters())
    return new

def count_scconv(module):
    return sum(1 for item in module.modules() if item.__class__.__name__ == 'ScConv')

def count_c3k2_scconv(module):
    return sum(1 for item in module.modules() if item.__class__.__name__ == 'C3k2_SCConv')

def sum_tensors(value):
    if torch.is_tensor(value):
        return value.float().sum()
    if isinstance(value, (list, tuple)):
        tensors = [sum_tensors(item) for item in value]
        return sum(tensors, torch.zeros((), device=tensors[0].device if tensors else None))
    if isinstance(value, dict):
        tensors = [sum_tensors(item) for item in value.values()]
        return sum(tensors, torch.zeros((), device=tensors[0].device if tensors else None))
    return torch.zeros(())

def build_custom_scconv_model():
    model_wrapper = YOLO(MODEL_NAME)
    replacements = []
    for spec in c3k2_specs:
        old = model_wrapper.model.model[spec['index']]
        if old.__class__.__name__ != 'C3k2':
            raise RuntimeError(f"Target {spec['level']} is {old.__class__.__name__}, not C3k2.")
        new = C3k2_SCConv(old)
        replacements.append((spec['level'], new.stats))
        model_wrapper.model.model[spec['index']] = copy_graph_metadata(new, old, spec['index'], spec['from'])
    return model_wrapper, replacements

custom_model, replacement_stats = build_custom_scconv_model()
source_state = base_model.model.state_dict()
target_state = custom_model.model.state_dict()
compatible_state = {
    key: value for key, value in source_state.items()
    if key in target_state and tuple(value.shape) == tuple(target_state[key].shape)
}
load_report = custom_model.model.load_state_dict(compatible_state, strict=False)
print('Replacement stats:', replacement_stats)
print('Compatible pretrained tensors:', len(compatible_state))
print('Missing custom tensors:', len(load_report.missing_keys))
print('C3k2_SCConv:', count_c3k2_scconv(custom_model.model), 'ScConv:', count_scconv(custom_model.model))
assert count_c3k2_scconv(custom_model.model) == C3K2_SCCONV_EXPECTED_COUNT
assert count_scconv(custom_model.model) > 0

device = torch.device('cuda:0')
custom_model.model.to(device)
detect_output_records = []
custom_detect_head = custom_model.model.model[detect_index]
detect_handle = custom_detect_head.register_forward_hook(
    lambda _module, _inputs, output: detect_output_records.append(output)
)
custom_model.model.eval()
try:
    with torch.no_grad():
        dummy = torch.randn(2, 3, 768, 768, device=device)
        dummy_output = custom_model.model(dummy)
except RuntimeError as error:
    if 'out of memory' in str(error).lower():
        raise RuntimeError('768 dummy forward failed with CUDA OOM before training.') from error
    raise
finally:
    detect_handle.remove()
assert detect_output_records, 'Detect head did not execute during 768 dummy forward.'
assert any(torch.is_tensor(item) for item in detect_output_records[0]) if isinstance(detect_output_records[0], (list, tuple)) else True
print('768 dummy forward: OK')

custom_model.model.train()
custom_model.model.zero_grad(set_to_none=True)
try:
    graph_output = custom_model.model(torch.randn(2, 3, 768, 768, device=device, requires_grad=True))
    graph_loss = sum_tensors(graph_output)
    assert graph_loss.requires_grad
    graph_loss.backward()
except RuntimeError as error:
    if 'out of memory' in str(error).lower():
        raise RuntimeError('768 backward preflight failed with CUDA OOM. E8 requires batch=16; stop without changing it.') from error
    raise
scconv_grad_ok = any(
    parameter.grad is not None
    for item in custom_model.model.modules()
    if item.__class__.__name__ == 'ScConv'
    for parameter in item.parameters()
)
print('SCConv gradient OK:', scconv_grad_ok)
assert scconv_grad_ok
custom_model.model.eval()

assert not EXPERIMENT_DIR.exists(), f'Experiment directory already exists: {EXPERIMENT_DIR}'
from ultralytics.models.yolo.detect import DetectionTrainer

custom_pytorch_model = custom_model.model

class CustomSCConvTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        return custom_pytorch_model

    def setup_model(self):
        self.model = custom_pytorch_model
        assert count_c3k2_scconv(self.model) == 2
        print('C3k2_SCConv INSIDE setup_model:', count_c3k2_scconv(self.model))
        return None

TRAIN_OVERRIDES = {
    'model': MODEL_NAME, 'data': str(DATA_YAML_PATH), **TRAIN_CONFIG,
    'project': str(OUTPUT_DIR), 'name': EXPERIMENT_NAME,
    'save': True, 'save_period': 10, 'plots': True, 'exist_ok': False,
    'val': True, 'split': VALIDATION_SPLIT,
}
trainer = CustomSCConvTrainer(overrides=TRAIN_OVERRIDES, _callbacks=custom_model.callbacks)
trainer.model = custom_pytorch_model
c3k2_scconv_before_train = count_c3k2_scconv(custom_pytorch_model)
c3k2_scconv_inside_trainer = count_c3k2_scconv(trainer.model)
print('C3k2_SCConv BEFORE TRAIN:', c3k2_scconv_before_train)
print('C3k2_SCConv INSIDE TRAINER:', c3k2_scconv_inside_trainer)
assert c3k2_scconv_before_train == c3k2_scconv_inside_trainer == 2

EXPECTED_BATCH = 16
FIXED_BATCH_OOM_MESSAGE = (
    'E8 stopped: imgsz=768 with batch=16 does not fit available VRAM. '
    'Do not retry automatically with a smaller batch.'
)

def enforce_fixed_batch(current_trainer):
    trainer_batch = getattr(current_trainer, 'batch_size', None)
    args_batch = getattr(current_trainer.args, 'batch', None)
    if trainer_batch is not None:
        trainer_batch = int(trainer_batch)
    if args_batch is not None:
        args_batch = int(args_batch)
    if trainer_batch != EXPECTED_BATCH or args_batch != EXPECTED_BATCH:
        raise RuntimeError(
            'E8 INVALID: batch size changed. '
            f'Expected batch={EXPECTED_BATCH}, '
            f'trainer.batch_size={trainer_batch}, '
            f'trainer.args.batch={args_batch}. '
            'Stop experiment instead of continuing with an auto-reduced batch. '
            + FIXED_BATCH_OOM_MESSAGE
        )

print('Initial trainer batch:', getattr(trainer, 'batch_size', None))
print('Initial args batch:', getattr(trainer.args, 'batch', None))
enforce_fixed_batch(trainer)
trainer.add_callback('on_train_batch_start', enforce_fixed_batch)

try:
    train_results = trainer.train()
except RuntimeError as error:
    if 'out of memory' in str(error).lower():
        raise RuntimeError(FIXED_BATCH_OOM_MESSAGE) from error
    raise
enforce_fixed_batch(trainer)
assert int(trainer.batch_size) == EXPECTED_BATCH
assert int(trainer.args.batch) == EXPECTED_BATCH
print('Final training batch confirmed:', trainer.batch_size)

EXPERIMENT_VERSION_FILE = EXPERIMENT_DIR / 'ultralytics_version.txt'
EXPERIMENT_VERSION_FILE.write_text(
    'ultralytics==' + ultralytics.__version__ + '\n'
    + 'model=' + MODEL_NAME + '\n'
    + 'imgsz=768\n'
    + 'c3k2_scconv_expected_count=2\n',
    encoding='utf-8',
)

def torch_load_checkpoint(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')

def extract_checkpoint_state(payload):
    if isinstance(payload, nn.Module):
        return payload.state_dict()
    if isinstance(payload, dict):
        for key in ('model', 'ema', 'state_dict'):
            value = payload.get(key)
            if isinstance(value, nn.Module):
                return value.state_dict()
            if isinstance(value, dict):
                tensors = {name: item for name, item in value.items() if torch.is_tensor(item)}
                if tensors:
                    return tensors
        tensors = {name: item for name, item in payload.items() if torch.is_tensor(item)}
        if tensors:
            return tensors
    raise TypeError('Checkpoint state dict not found.')

def validate_checkpoint_integrity(model, target, compatible, report=None, source_state=None):
    target_count = len(target)
    compatible_count = len(compatible)
    transfer_ratio = compatible_count / target_count if target_count > 0 else 0.0
    missing_keys = list(report.missing_keys) if report is not None else []
    unexpected_keys = set(report.unexpected_keys) if report is not None else set()
    if source_state is not None:
        unexpected_keys.update(name for name in source_state if name not in target)
    unexpected_keys = sorted(unexpected_keys)
    print('Checkpoint transfer:', compatible_count, '/', target_count, '=', f'{transfer_ratio:.2%}')
    print('Missing keys:', len(missing_keys))
    print('Unexpected keys:', len(unexpected_keys))
    print('First missing keys:', missing_keys[:20])
    print('First unexpected keys:', unexpected_keys[:20])
    scconv_module_names = [
        name for name, module in model.model.named_modules()
        if module.__class__.__name__ in {'C3k2_SCConv', 'ScConv', 'SRU', 'CRU'}
    ]
    critical_missing = [
        key for key in missing_keys
        if any(key.startswith(name + '.') for name in scconv_module_names if name)
    ]
    print('Critical SCConv missing keys:', critical_missing[:20])
    assert transfer_ratio >= 0.95, (
        'Checkpoint loading is incomplete. '
        f'Transfer ratio only {transfer_ratio:.2%}.'
    )
    assert not critical_missing, (
        'Missing critical SCConv weights: ' f'{critical_missing[:20]}'
    )
    assert count_c3k2_scconv(model.model) == 2
    assert count_scconv(model.model) > 0
    return {
        'target_count': target_count,
        'compatible_count': compatible_count,
        'transfer_ratio': transfer_ratio,
        'missing_keys': missing_keys,
        'unexpected_keys': unexpected_keys,
        'critical_missing': critical_missing,
    }

def load_custom_scconv_checkpoint(path):
    try:
        direct = YOLO(str(path))
        if count_c3k2_scconv(direct.model) == 2 and count_scconv(direct.model) > 0:
            direct_target = direct.model.state_dict()
            direct_integrity = validate_checkpoint_integrity(
                direct, direct_target, direct_target
            )
            return direct, 'direct', direct_integrity
    except Exception as direct_error:
        print('Direct custom checkpoint load unavailable:', direct_error)
    rebuilt, _ = build_custom_scconv_model()
    state = extract_checkpoint_state(torch_load_checkpoint(str(path)))
    if state and all(name.startswith('module.') for name in state):
        state = {name[7:]: value for name, value in state.items()}
    target = rebuilt.model.state_dict()
    compatible = {
        name: value for name, value in state.items()
        if name in target and tuple(value.shape) == tuple(target[name].shape)
    }
    if not compatible:
        raise RuntimeError('No compatible custom checkpoint tensors.')
    report = rebuilt.model.load_state_dict(compatible, strict=False)
    integrity = validate_checkpoint_integrity(
        rebuilt, target, compatible, report=report, source_state=state
    )
    return rebuilt, 'rebuilt', integrity

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(BEST_MODEL_PATH)
best_model, checkpoint_load_mode, e8_checkpoint_integrity = load_custom_scconv_checkpoint(BEST_MODEL_PATH)
c3k2_scconv_count_best = count_c3k2_scconv(best_model.model)
scconv_count_best = count_scconv(best_model.model)
print('C3k2_SCConv IN BEST MODEL:', c3k2_scconv_count_best)
print('ScConv IN BEST MODEL:', scconv_count_best, 'load mode:', checkpoint_load_mode)
assert c3k2_scconv_count_best == 2 and scconv_count_best > 0

## 3. Validation metrics, confusion statistics, curves, complexity, and E5 comparison

E8 is selected using `val` only. The comparison reports the no-helmet class
first, while keeping the resolution-related latency and VRAM cost explicit.

In [ ]:
def safe_f1(precision, recall):
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

def extract_metrics(result, label):
    summary = {
        'Precision': float(result.box.mp),
        'Recall': float(result.box.mr),
        'mAP50': float(result.box.map50),
        'mAP50-95': float(result.box.map),
    }
    summary['F1'] = safe_f1(summary['Precision'], summary['Recall'])
    summary['by_class'] = {}
    for class_id, class_name in enumerate(CLASS_NAMES):
        precision, recall, ap50, ap5095 = result.box.class_result(class_id)
        values = {
            'Precision': float(precision), 'Recall': float(recall),
            'AP50': float(ap50), 'AP50-95': float(ap5095),
        }
        values['F1'] = safe_f1(values['Precision'], values['Recall'])
        summary['by_class'][class_name] = values
    print(label, summary)
    return summary

validation_metrics = best_model.val(
    data=str(DATA_YAML_PATH), split=VALIDATION_SPLIT, imgsz=768, device=0, plots=True,
)
validation_summary = extract_metrics(validation_metrics, 'E8 validation')
no_helmet_metrics = validation_summary['by_class']['rider_without_helmet']
print('E8 rider_without_helmet metrics:', no_helmet_metrics)
test_metrics = None
test_summary = None

def confusion_array(result):
    matrix = getattr(getattr(result, 'confusion_matrix', None), 'matrix', None)
    matrix = None if matrix is None else np.asarray(matrix)
    expected_shape = (len(CLASS_NAMES) + 1, len(CLASS_NAMES) + 1)
    return matrix if matrix is not None and matrix.shape == expected_shape else None

def confusion_statistics(result):
    matrix = confusion_array(result)
    if matrix is None:
        return None
    labels = CLASS_NAMES + ['background']
    index = {label: i for i, label in enumerate(labels)}
    nohelmet = index['rider_without_helmet']
    column_total = matrix[:, nohelmet].sum()
    return {
        'NoHelmet -> NoHelmet': float(matrix[nohelmet, nohelmet]),
        'NoHelmet -> Motorcycle': float(matrix[index['motorcycle'], nohelmet]),
        'NoHelmet -> WithHelmet': float(matrix[index['rider_with_helmet'], nohelmet]),
        'NoHelmet -> Background': float(matrix[index['background'], nohelmet]),
        'Background -> NoHelmet': float(matrix[nohelmet, index['background']]),
        'NoHelmet normalized diagonal': float(matrix[nohelmet, nohelmet] / column_total) if column_total else 0.0,
    }

e8_confusion_stats = confusion_statistics(validation_metrics)
if e8_confusion_stats is not None:
    display(pd.DataFrame({'Statistic': list(e8_confusion_stats), 'Value': list(e8_confusion_stats.values())}))

plot_specs = [
    ('results.png', 'Training results'),
    ('confusion_matrix.png', 'Confusion matrix'),
    ('confusion_matrix_normalized.png', 'Normalized confusion matrix'),
    ('BoxPR_curve.png', 'Box PR curve'),
    ('BoxF1_curve.png', 'Box F1 curve'),
    ('BoxP_curve.png', 'Box Precision curve'),
    ('BoxR_curve.png', 'Box Recall curve'),
]
validation_dir = Path(str(validation_metrics.save_dir))
for filename, title in plot_specs:
    paths = list(validation_dir.rglob(filename)) if validation_dir.exists() else []
    if paths:
        path = max(paths, key=lambda item: item.stat().st_mtime)
        plt.figure(figsize=(16, 10) if filename == 'results.png' else (10, 8))
        plt.imshow(Image.open(path))
        plt.axis('off')
        plt.title(title)
        plt.show()

valid_img_dir = Path('/content/DetectedHelmets/DataSet2/DataSet2/valid/images')
valid_images = sorted(
    str(path) for pattern in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
    for path in valid_img_dir.glob(pattern)
)
assert valid_images

def extract_gflops(info):
    if isinstance(info, dict):
        for key in ('GFLOPs', 'gflops', 'flops'):
            if key in info:
                try:
                    return float(info[key])
                except (TypeError, ValueError):
                    pass
    values = list(info) if isinstance(info, (list, tuple)) else []
    for value in reversed(values):
        try:
            return float(value)
        except (TypeError, ValueError):
            pass
    return None

def benchmark(model, images, imgsz):
    torch.cuda.empty_cache()
    model.predict(source=images[0], imgsz=imgsz, device=0, save=False, verbose=False)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    elapsed = []
    for image in images:
        torch.cuda.synchronize()
        start = time.perf_counter()
        model.predict(source=image, imgsz=imgsz, device=0, save=False, verbose=False)
        torch.cuda.synchronize()
        elapsed.append(time.perf_counter() - start)
    milliseconds = float(np.mean(elapsed) * 1000)
    peak_vram_gb = float(torch.cuda.max_memory_allocated() / 1024 ** 3)
    return milliseconds, 1000 / milliseconds, peak_vram_gb

def resolve_e5_checkpoint(primary_path):
    if primary_path.exists():
        return primary_path
    search_root = OUTPUT_DIR.parent
    if not search_root.exists():
        return None
    candidates = sorted(
        path for path in search_root.rglob('best.pt')
        if E5_EXPERIMENT_NAME in str(path)
    )
    if len(candidates) == 1:
        print('Using discovered E5 checkpoint:', candidates[0])
        return candidates[0]
    if candidates:
        print('Ambiguous E5 checkpoint candidates:', candidates)
    return None

e5_checkpoint_path = resolve_e5_checkpoint(E5_BEST_MODEL_PATH)
e5_model = None
e5_checkpoint_load_mode = None
e5_checkpoint_integrity = None
e5_validation_metrics = None
e5_summary = None
e5_confusion_stats = None
if e5_checkpoint_path is not None:
    e5_model, e5_checkpoint_load_mode, e5_checkpoint_integrity = load_custom_scconv_checkpoint(e5_checkpoint_path)
    e5_validation_metrics = e5_model.val(
        data=str(DATA_YAML_PATH), split=VALIDATION_SPLIT, imgsz=640, device=0, plots=False,
    )
    e5_summary = extract_metrics(e5_validation_metrics, 'E5 validation at 640')
    e5_confusion_stats = confusion_statistics(e5_validation_metrics)
else:
    print('WARNING: E5 checkpoint not found; E5 comparison will be skipped.')
    print('Expected path:', E5_BEST_MODEL_PATH)
    print('E8 validation and E8-only complexity reporting will continue.')
speed_images = valid_images[:min(50, len(valid_images))]
e8_ms, e8_fps, e8_peak_vram = benchmark(best_model, speed_images, imgsz=768)
complexity_rows = [{
    'Experiment': 'E8 SCConv @768',
    'Parameters': sum(parameter.numel() for parameter in best_model.model.parameters()),
    'GFLOPs': extract_gflops(best_model.info(verbose=True, imgsz=768)),
    'Inference ms/image': e8_ms, 'FPS': e8_fps, 'GPU VRAM peak (GB)': e8_peak_vram,
}]
if e5_model is not None:
    e5_ms, e5_fps, e5_peak_vram = benchmark(e5_model, speed_images, imgsz=640)
    complexity_rows.insert(0, {
        'Experiment': 'E5 SCConv @640',
        'Parameters': sum(parameter.numel() for parameter in e5_model.model.parameters()),
        'GFLOPs': extract_gflops(e5_model.info(verbose=True, imgsz=640)),
        'Inference ms/image': e5_ms, 'FPS': e5_fps, 'GPU VRAM peak (GB)': e5_peak_vram,
    })
complexity = pd.DataFrame(complexity_rows)
display(complexity)

def comparison_values(summary, confusion_stats):
    nohelmet = summary['by_class']['rider_without_helmet']
    motorcycle = summary['by_class']['motorcycle']
    withhelmet = summary['by_class']['rider_with_helmet']
    return {
        'Overall Precision': summary['Precision'],
        'Overall Recall': summary['Recall'],
        'Overall F1': summary['F1'],
        'mAP50': summary['mAP50'],
        'mAP50-95': summary['mAP50-95'],
        'Motorcycle AP50': motorcycle['AP50'],
        'WithHelmet AP50': withhelmet['AP50'],
        'NoHelmet AP50': nohelmet['AP50'],
        'NoHelmet Precision': nohelmet['Precision'],
        'NoHelmet Recall': nohelmet['Recall'],
        'NoHelmet F1': nohelmet['F1'],
        'NoHelmet AP50-95': nohelmet['AP50-95'],
        'NoHelmet normalized diagonal': confusion_stats['NoHelmet normalized diagonal'] if confusion_stats else np.nan,
        'NoHelmet -> WithHelmet': confusion_stats['NoHelmet -> WithHelmet'] if confusion_stats else np.nan,
        'NoHelmet -> Background': confusion_stats['NoHelmet -> Background'] if confusion_stats else np.nan,
        'Background -> NoHelmet': confusion_stats['Background -> NoHelmet'] if confusion_stats else np.nan,
    }

e8_values = comparison_values(validation_summary, e8_confusion_stats)
if e5_summary is not None:
    e5_values = comparison_values(e5_summary, e5_confusion_stats)
    comparison = pd.DataFrame({
        'Metric': list(e5_values),
        'E5 SCConv @640': list(e5_values.values()),
        'E8 SCConv @768': [e8_values[key] for key in e5_values],
    })
    comparison['Delta'] = comparison['E8 SCConv @768'] - comparison['E5 SCConv @640']
    display(comparison)
    priority_metrics = ['NoHelmet AP50', 'NoHelmet Recall', 'NoHelmet normalized diagonal']
    priority_improvements = {metric: e8_values[metric] > e5_values[metric] for metric in priority_metrics}
    overall_mAP50_drop = e8_values['mAP50'] < e5_values['mAP50'] - 0.01
    e8_keep = any(priority_improvements.values()) and not overall_mAP50_drop
    print('Priority improvements:', priority_improvements)
    print('Overall mAP50 significant drop:', overall_mAP50_drop)
    print('E8 decision:', 'KEEP E8' if e8_keep else 'REJECT E8')
else:
    display(pd.DataFrame({'Metric': list(e8_values), 'E8 SCConv @768': list(e8_values.values())}))
    e8_keep = None
    print('E8 decision deferred: E5 checkpoint is unavailable.')
print('Targets for interpretation only: NoHelmet AP50 >= 0.82, normalized diagonal >= 0.78, overall mAP50 >= 0.90, mAP50-95 >= E5.')
print('Paper note: E5@640 and E8@768 are not equal-compute comparisons; a YOLO26s baseline@768 is needed before attributing all gains to SCConv.')

## 4. Final audit

The final cell is intentionally strict about the experimental protocol and
intentionally does not assert any target metric value. Metric thresholds are
reported for interpretation, not hard-coded as results.

In [ ]:
assert ultralytics.__version__ == '8.4.155'
assert CLASS_NAMES == EXPECTED_CLASS_NAMES
assert DATA_YAML_PATH == Path('/content/DetectedHelmets/DataSet2/DataSet2/data.yaml')
assert MODEL_NAME == 'yolo26s.pt'
assert TRAIN_CONFIG['imgsz'] == 768
assert TRAIN_CONFIG['epochs'] == 50 and TRAIN_CONFIG['patience'] == 15
assert TRAIN_CONFIG['batch'] == 16 and TRAIN_CONFIG['device'] == 0 and TRAIN_CONFIG['workers'] == 4
assert TRAIN_CONFIG['optimizer'] == 'auto' and TRAIN_CONFIG['seed'] == 0 and TRAIN_CONFIG['deterministic'] is True
assert TRAIN_CONFIG['pretrained'] is True and TRAIN_CONFIG['close_mosaic'] == 10 and TRAIN_CONFIG['amp'] is True
assert TRAIN_CONFIG['hsv_h'] == 0.015 and TRAIN_CONFIG['hsv_s'] == 0.7 and TRAIN_CONFIG['hsv_v'] == 0.4
assert TRAIN_CONFIG['degrees'] == 0.0 and TRAIN_CONFIG['translate'] == 0.1 and TRAIN_CONFIG['scale'] == 0.5
assert TRAIN_CONFIG['shear'] == 0.0 and TRAIN_CONFIG['perspective'] == 0.0
assert TRAIN_CONFIG['flipud'] == 0.0 and TRAIN_CONFIG['fliplr'] == 0.5
assert TRAIN_CONFIG['mosaic'] == 1.0 and TRAIN_CONFIG['mixup'] == 0.0 and TRAIN_CONFIG['cutmix'] == 0.0 and TRAIN_CONFIG['copy_paste'] == 0.0
assert VALIDATION_SPLIT == 'val'
assert trainer.model is custom_pytorch_model
assert int(trainer.batch_size) == EXPECTED_BATCH
assert int(trainer.args.batch) == EXPECTED_BATCH
assert TRAIN_CONFIG['batch'] == EXPECTED_BATCH == 16
assert c3k2_scconv_before_train == c3k2_scconv_inside_trainer == c3k2_scconv_count_best == 2
assert scconv_grad_ok and scconv_count_best > 0
assert e8_checkpoint_integrity['transfer_ratio'] >= 0.95
assert not e8_checkpoint_integrity['critical_missing']
if e5_checkpoint_integrity is not None:
    assert e5_checkpoint_integrity['transfer_ratio'] >= 0.95
    assert not e5_checkpoint_integrity['critical_missing']
else:
    print('E5 checkpoint unavailable; E5 comparison was skipped.')
assert test_metrics is None and test_summary is None
assert EXPERIMENT_NAME == 'helmet_yolo26s_scconv_768_dataset2'
assert BEST_MODEL_PATH == Path('/content/drive/MyDrive/DetectedHelmets/runs/train/helmet_yolo26s_scconv_768_dataset2/weights/best.pt')
forbidden = {'P2', 'iAFF', 'MultiSEAM', 'DySample', 'SEAM', 'CBAM', 'ECA', 'AFPN', 'ADown'}
custom_names = {item.__class__.__name__ for item in best_model.model.modules()}
assert not custom_names.intersection(forbidden)
assert EXPERIMENT_VERSION_FILE.exists()
print('Final audit passed.')
print('Ultralytics:', ultralytics.__version__)
print('C3k2_SCConv before/train/best:', c3k2_scconv_before_train, c3k2_scconv_inside_trainer, c3k2_scconv_count_best)
print('SCConv in best model:', scconv_count_best)
print('E8 fixed batch verified: 16')
print('E8 checkpoint integrity verified')
print('Validation only; test disabled; output:', BEST_MODEL_PATH)
print('E8 changes only the input resolution from 640 to 768 relative to E5.')